In [ ]:
"""
    VIDEO TEST
"""

import cv2
import re
import torch
import pandas as pd

from PIL import Image
from ultralytics import YOLO


# ============================================================
# CONFIGURACIÓN
# ============================================================

USE_SMOLVLM = False

VIDEO = "C0142.MP4"

FRAME_SKIP = 2          # YOLO matrículas: cada 2 frames
OCR_SECONDS = 0.25      # OCR: cada cuarto de segundo de vídeo
MAX_INTENTOS = 3        # Intentos máximos por vehículo

MIN_PLATE_WIDTH = 80    # Ancho mínimo de matrícula
MIN_PLATE_HEIGHT = 20   # Alto mínimo de matrícula

PLATE_PATTERN = r'^\d{4}[BCDFGHJKLMNPRSTVWXYZ]{3}$'

# Modelo de matrículas
PLATE_MODEL_PATH = r"lp_weights\yolo11n_960_b2_scale08\best.pt"

# Vídeo
cap = cv2.VideoCapture(VIDEO)

fps = cap.get(cv2.CAP_PROP_FPS)

OCR_INTERVAL = max(1, int(fps * OCR_SECONDS))

print(f"FPS del vídeo: {fps}")
print(f"Intervalo YOLO matrículas: {FRAME_SKIP} frames")
print(f"Intervalo OCR: {OCR_INTERVAL} frames")


# ============================================================
# INICIALIZACIÓN DE OCR
# ============================================================

if USE_SMOLVLM:
    from transformers import AutoProcessor, AutoModelForVision2Seq
    

    MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    processor = AutoProcessor.from_pretrained(MODEL_ID)

    model = AutoModelForVision2Seq.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
    ).to(DEVICE)

    model.eval()

else:
    import easyocr

    reader = easyocr.Reader(
        ['en'],
        gpu=torch.cuda.is_available()
    )


# ============================================================
# FUNCIONES OCR
# ============================================================

def smolvlm_get_text(box, frame, processor, model, device):

    mx1, my1, mx2, my2 = map(
        int, box.xyxy[0].cpu().numpy()
    )

    plate_crop = frame[my1:my2, mx1:mx2]

    texto = None
    confianza_ocr = None

    if plate_crop.size > 0:
        roi_rgb = cv2.cvtColor(
            plate_crop,
            cv2.COLOR_BGR2RGB
        )

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {
                        "type": "text",
                        "text": (
                            "Read the vehicle license plate. "
                            "Return only the license plate characters, "
                            "without spaces or explanations."
                        )
                    }
                ]
            }
        ]

        prompt = processor.apply_chat_template(
            messages,
            add_generation_prompt=True
        )

        inputs = processor(
            text=prompt,
            images=Image.fromarray(roi_rgb),
            return_tensors="pt"
        ).to(device)

        with torch.inference_mode():

            generated_ids = model.generate(
                **inputs,
                max_new_tokens=20,
                do_sample=False
            )

        generated_ids = generated_ids[
            :, inputs.input_ids.shape[1]:
        ]

        texto = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )[0].strip().upper()

        texto = re.sub(r'[^A-Z0-9]', '', texto)

        if not texto:
            texto = None

    return {
        "confianza_matricula": float(box.conf[0]),
        "bbox": (mx1, my1, mx2, my2),
        "texto": texto,
        "confianza_ocr": confianza_ocr
    }


def easyocr_get_text(box, frame, reader):
    mx1, my1, mx2, my2 = map(
        int, box.xyxy[0].cpu().numpy()
    )

    plate_crop = frame[my1:my2, mx1:mx2]

    texto = None
    confianza_ocr = None

    if plate_crop.size > 0:
        ocr_results = reader.readtext(
            plate_crop,
            detail=1,
            paragraph=False,
            allowlist='0123456789BCDFGHJKLMNPRSTVWXYZ'
        )

        if ocr_results:
            mejor_resultado = max(
                ocr_results,
                key=lambda x: x[2]
            )

            texto = re.sub(
                r'[^A-Z0-9]',
                '',
                mejor_resultado[1].upper()
            )

            confianza_ocr = float(mejor_resultado[2])

            if not texto:
                texto = None

    return {
        "confianza_matricula": float(box.conf[0]),
        "bbox": (mx1, my1, mx2, my2),
        "texto": texto,
        "confianza_ocr": confianza_ocr
    }

# =====================
# CARGAR MODELOS YOLO
# =====================

model_coco = YOLO("yolo11n.pt")

model_custom = YOLO(PLATE_MODEL_PATH)

# ============================
# VARIABLES DE PROCESAMIENTO
# ============================

datos_csv = []

fotograma = 0

paused = False

# Información almacenada por vehículo (tracking ID)
vehiculos_estado = {}


# ==================
# BUCLE PRINCIPAL
# ==================

while cap.isOpened():

    if not paused:

        ret, frame = cap.read()

        if not ret:
            break

        fotograma += 1

        # -------------------------------------
        # 1. YOLO COCO: TODOS LOS FOTOGRAMAS
        # -------------------------------------

        results_coco = model_coco.track(
            frame,
            persist=True,
            verbose=False,
            classes=[0, 1, 2, 3, 5, 7] # personas, biciclietas, coches, motos, autobuses y camiones
        )

        vehicles = []

        for result in results_coco:
            for box in result.boxes:

                x1, y1, x2, y2 = map(
                    int, box.xyxy[0].cpu().numpy()
                )

                vehicles.append({
                    "tipo": model_coco.names[int(box.cls[0])],
                    "confianza": float(box.conf[0]),
                    "tracking_id": (
                        int(box.id[0])
                        if box.id is not None
                        else None
                    ),
                    "bbox": (x1, y1, x2, y2)
                })


        # ----------------------------------------
        # 2. YOLO MATRÍCULAS: CADA 5 FOTOGRAMAS
        # ----------------------------------------

        matriculas = []

        ejecutar_yolo_placas = fotograma % FRAME_SKIP == 0

        if ejecutar_yolo_placas:
            results_custom = model_custom.predict(
                frame,
                verbose=False
            )

            for result in results_custom:
                for box in result.boxes:

                    mx1, my1, mx2, my2 = map(
                        int, box.xyxy[0].cpu().numpy()
                    )

                    matriculas.append({
                        "box": box,
                        "bbox": (mx1, my1, mx2, my2),
                        "confianza_matricula": float(box.conf[0])
                    })


        # --------------------------------------
        # 3. ASOCIAR MATRÍCULAS CON VEHÍCULOS
        # --------------------------------------

        if ejecutar_yolo_placas:

            for matricula in matriculas:

                mx1, my1, mx2, my2 = matricula["bbox"]

                cx = (mx1 + mx2) / 2
                cy = (my1 + my2) / 2

                candidatos = []

                for vehiculo in vehicles:
                    x1, y1, x2, y2 = vehiculo["bbox"]

                    if x1 <= cx <= x2 and y1 <= cy <= y2:
                        area = (x2 - x1) * (y2 - y1)
                        candidatos.append((area, vehiculo))

                if candidatos:
                    candidatos.sort(key=lambda x: x[0])

                    vehiculo_asociado = candidatos[0][1]

                    tracking_id = vehiculo_asociado["tracking_id"]

                    if tracking_id is None:
                        continue

                    # Crear estado si es un vehículo nuevo
                    if tracking_id not in vehiculos_estado:

                        vehiculos_estado[tracking_id] = {
                            "texto": None,
                            "confianza_ocr": None,
                            "bbox_matricula": None,
                            "confianza_matricula": None,
                            "ultimo_ocr": -OCR_INTERVAL,
                            "intentos": 0,
                            "ultimo_visto": fotograma
                        }

                    estado = vehiculos_estado[tracking_id]
                    estado["ultimo_visto"] = fotograma
                    estado["bbox_matricula"] = matricula["bbox"]
                    estado["confianza_matricula"] = (matricula["confianza_matricula"])

                    # ------------------------------------
                    # 4. CONTROL DE FRECUENCIA DEL OCR
                    # ------------------------------------

                    mx1, my1, mx2, my2 = matricula["bbox"]

                    ancho = mx2 - mx1
                    alto = my2 - my1

                    tamano_suficiente = (
                        ancho >= MIN_PLATE_WIDTH
                        and alto >= MIN_PLATE_HEIGHT
                    )

                    intervalo_cumplido = fotograma - estado["ultimo_ocr"] >= OCR_INTERVAL

                    quedan_intentos = estado["intentos"] < MAX_INTENTOS

                    matricula_reconocida = estado["texto"] is not None

                    if (
                        not matricula_reconocida
                        and tamano_suficiente
                        and intervalo_cumplido
                        and quedan_intentos
                    ):

                        print(
                            f"OCR vehículo {tracking_id} "
                            f"| Intento {estado['intentos'] + 1}"
                        )

                        if USE_SMOLVLM:
                            resultado = smolvlm_get_text(
                                matricula["box"],
                                frame,
                                processor,
                                model,
                                DEVICE
                            )

                        else:
                            resultado = easyocr_get_text(
                                matricula["box"],
                                frame,
                                reader
                            )

                        estado["ultimo_ocr"] = fotograma
                        estado["intentos"] += 1
                        texto = resultado["texto"]

                        # Validar formato de matrícula española
                        if texto and re.fullmatch(PLATE_PATTERN, texto):
                            estado["texto"] = texto
                            estado["confianza_ocr"] = resultado["confianza_ocr"]

                            print(
                                f"Matrícula reconocida: "
                                f"{texto} (ID {tracking_id})"
                            )

        # ---------------------------------------------
        # 5. LIMPIAR VEHÍCULOS QUE HAN DESAPARECIDO
        # ---------------------------------------------

        ids_activos = {
            v["tracking_id"]
            for v in vehicles
            if v["tracking_id"] is not None
        }

        # Conservar temporalmente los vehículos desaparecidos
        for tracking_id in list(vehiculos_estado.keys()):

            ultimo_visto = vehiculos_estado[
                tracking_id
            ]["ultimo_visto"]

            if (
                fotograma - ultimo_visto > fps * 5
                and tracking_id not in ids_activos
            ):

                del vehiculos_estado[tracking_id]


        # ----------------------------
        # 6. GENERAR REGISTROS CSV
        # ----------------------------

        for vehiculo in vehicles:
            tracking_id = vehiculo["tracking_id"]

            estado = vehiculos_estado.get(tracking_id)

            if estado is not None:

                mx1, my1, mx2, my2 = (
                    estado["bbox_matricula"]
                    if estado["bbox_matricula"] is not None
                    else (None, None, None, None)
                )

                texto = estado["texto"]
                confianza_ocr = estado["confianza_ocr"]
                matricula_en_su_caso = (estado["bbox_matricula"] is not None)

            else:
                mx1 = my1 = mx2 = my2 = None
                texto = None
                confianza_ocr = None
                matricula_en_su_caso = False

            x1, y1, x2, y2 = vehiculo["bbox"]

            datos_csv.append({
                "fotograma": fotograma,
                "tipo_objeto": vehiculo["tipo"],
                "confianza_objeto": vehiculo["confianza"],
                "identificador_tracking": tracking_id,
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "matricula_en_su_caso": matricula_en_su_caso,
                "confianza_ocr": confianza_ocr,
                "mx1": mx1,
                "my1": my1,
                "mx2": mx2,
                "my2": my2,
                "texto_matricula": texto
            })


        # -------------------------
        # 7. DIBUJAR DETECCIONES
        # -------------------------

        frame = results_coco[0].plot(img=frame)

        # Dibujar matrículas y texto reconocido
        if ejecutar_yolo_placas:

            for matricula in matriculas:

                mx1, my1, mx2, my2 = matricula["bbox"]

                # Buscar el vehículo asociado
                cx = (mx1 + mx2) / 2
                cy = (my1 + my2) / 2

                texto = None

                for vehiculo in vehicles:

                    x1, y1, x2, y2 = vehiculo["bbox"]

                    if x1 <= cx <= x2 and y1 <= cy <= y2:

                        tracking_id = vehiculo["tracking_id"]

                        if tracking_id in vehiculos_estado:
                            texto = vehiculos_estado[tracking_id]["texto"]

                        break

                # Bounding box de la matrícula
                cv2.rectangle(
                    frame,
                    (mx1, my1),
                    (mx2, my2),
                    (0, 255, 0),
                    2
                )

                # Texto debajo de la bounding box
                if texto:
                    cv2.putText(
                        frame,
                        texto,
                        (mx1, my2 + 25),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.8,
                        (0, 255, 0),
                        2,
                        cv2.LINE_AA
                    )

    # -------------------------
    # 8. CONTROLES DEL VÍDEO
    # -------------------------

    cv2.imshow("Deteccion combinada", frame)

    key = cv2.waitKey(1) & 0xFF

    if key == ord('q'):
        break

    elif key == ord('p'):
        paused = not paused


# =================
# EXPORTAR CSV
# =================

df = pd.DataFrame(datos_csv)

dir = "detecciones"

if USE_SMOLVLM:
    csv_file = f"{dir}/detecciones_smolVLM.csv"
    csv_filtrado = f"{dir}/detecciones_matricula_smolVLM.csv"
else:
    csv_file = f"{dir}/detecciones_easyOCR.csv"
    csv_filtrado = f"{dir}/detecciones_matricula_easyOCR.csv"

# Todas las detecciones
df.to_csv(
    csv_file,
    index=False,
    encoding="utf-8-sig"
)

df_matriculas = df[
    df["texto_matricula"].notna() &
    (df["texto_matricula"].str.strip() != "")
]

# solo las detecciones con matriculas
df_matriculas.to_csv(
    csv_filtrado,
    index=False,
    encoding="utf-8-sig"
)

cap.release()

cv2.destroyAllWindows()

print("\nProcesamiento finalizado.")
print(f"Fotogramas procesados: {fotograma}")
print(f"Registros CSV: {len(df)}")


In [ ]:
"""
    COMPARATIVA DE OCRs
"""

# smolVLM
from transformers import AutoProcessor, AutoModelForVision2Seq
from PIL import Image
model_id = "HuggingFaceTB/SmolVLM-256M-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
processor = AutoProcessor.from_pretrained(model_id)

model = AutoModelForVision2Seq.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

model.eval()

# easyOCR
import easyocr

reader = easyocr.Reader(
    ['en'],
    gpu=torch.cuda.is_available()
)

def easyocr_get_text(image, reader):

    texto = None
    confianza_ocr = None

    resultados = reader.readtext(
        image,
        detail=1,
        paragraph=False,
        allowlist='0123456789BCDFGHJKLMNPRSTVWXYZ'
    )

    if resultados:
        mejor_resultado = max(resultados, key=lambda x: x[2])

        texto = mejor_resultado[1].upper()
        confianza_ocr = float(mejor_resultado[2])

    texto = re.sub(r'[^A-Z0-9]', '', texto) if texto else None

    return {
        "texto": texto,
        "confianza_ocr": confianza_ocr
    }


def smolvlm_get_text(image, processor, model, device):

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_pil = Image.fromarray(image_rgb)

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_pil},
                {
                    "type": "text",
                    "text": "Read the vehicle license plate. Return only the license plate characters, without spaces or explanations."
                }
            ]
        }
    ]

    prompt = processor.apply_chat_template(
        messages,
        add_generation_prompt=True
    )

    inputs = processor(
        text=prompt,
        images=[image_pil],
        return_tensors="pt"
    ).to(device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False
        )

    generated_ids = generated_ids[:, inputs["input_ids"].shape[1]:]

    texto = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0].upper()

    texto = re.sub(r'[^A-Z0-9]', '', texto)

    return texto if texto else None


from pathlib import Path
import cv2
import re
import pandas as pd

import time

dataset_dir = Path("dataset_ocr")

datos = []

for img_path in dataset_dir.glob("*.png"):
    # Leer imagen
    image = cv2.imread(str(img_path))

    # Texto real obtenido del nombre
    texto_real = img_path.stem.upper()

    # EasyOCR
    inicio_easyocr = time.perf_counter()
    resultado_easy = easyocr_get_text(image, reader)
    fin_easyocr = time.perf_counter()
    tiempo_easyocr = fin_easyocr - inicio_easyocr

    # SmolVLM
    inicio_smolvlm = time.perf_counter()
    resultado_smol = smolvlm_get_text(
        image, processor, model, device
    )
    fin_smolvlm = time.perf_counter()
    tiempo_smolvlm = fin_smolvlm - inicio_smolvlm

    datos.append({
        "imagen": img_path.name,
        "texto_real": texto_real,
        "easyocr": resultado_easy["texto"],
        "tiempo_easyocr": tiempo_easyocr,
        "smolvlm": resultado_smol,
        "tiempo_smolvlm": tiempo_smolvlm,
    })

df = pd.DataFrame(datos)
df.to_csv("comparativa_ocr.csv", index=False)

print(df)

d:\anaconda3\envs\VC_P4\lib\site-packages\transformers\models\auto\modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


           imagen texto_real   easyocr  tiempo_easyocr  \
0     0019GVM.png    0019GVM      0019       -0.186800   
1     0041LZN.png    0041LZN   00L1LZM       -0.034453   
2     0063HSJ.png    0063HSJ   0063HSJ       -0.019635   
3     0083CPC.png    0083CPC   0083CPC       -0.062332   
4     0129KMK.png    0129KMK   0129KMK       -0.046255   
..            ...        ...       ...             ...   
201  GC2670CM.png   GC2670CM  GC2670CM       -0.026319   
202  GC4370BV.png   GC4370BV  GCL370BV       -0.026117   
203  GC4905BV.png   GC4905BV  CCZ90SBV       -0.026670   
204  GC8300CB.png   GC8300CB  GC8300CB       -0.044238   
205  GR5101AT.png   GR5101AT      None       -0.008697   

                                             smolvlm  tiempo_smolvlm  
0                                            0019GVM       -1.188983  
1                                            0041LZN       -1.303122  
2                                            0063HSJ       -0.971895  
3    THISISTHECODEF